In [1]:
import subprocess
import json
import os
import numpy as np

In [2]:
import subprocess
import json
from pathlib import Path
import numpy as np

def get_video_info(path):
    """Extrait les métadonnées via ffprobe."""
    cmd = [
        "ffprobe", "-v", "error", "-select_streams", "v:0",
        "-show_entries", "format=bit_rate:stream=codec_name,avg_frame_rate,width,height",
        "-of", "json", str(path)
    ]
    result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    return json.loads(result.stdout)

def analyze_folder_recursive(folder_path):
    """Parcourt récursivement le dossier pour analyser les fichiers MP4."""
    bitrates = []
    codecs = {}
    file_count = 0
    
    # .rglob("*") cherche récursivement
    # On peut filtrer sur .mp4 (insensible à la casse)
    for file_path in Path(folder_path).rglob("*"):
        if file_path.suffix.lower() == '.mp4':
            try:
                info = get_video_info(file_path)
                
                # Extraction du bitrate (peut être dans 'format' ou 'streams')
                br = info.get('format', {}).get('bit_rate')
                if br:
                    bitrates.append(int(br))
                
                # Extraction du codec
                if 'streams' in info and len(info['streams']) > 0:
                    codec = info['streams'][0].get('codec_name', 'inconnu')
                    codecs[codec] = codecs.get(codec, 0) + 1
                
                file_count += 1
            except Exception as e:
                print(f"Erreur sur {file_path.name}: {e}")
                continue
                
    return {
        "fichiers_analyses": file_count,
        "avg_bitrate_kbps": (np.mean(bitrates) / 1000) if bitrates else 0,
        "std_bitrate_kbps": (np.std(bitrates) / 1000) if bitrates else 0,
        "codecs": codecs
    }

# --- UTILISATION ---
# Remplacez par vos vrais chemins
# stats_vrais = analyze_folder_recursive("donnees/vrais")
# stats_faux = analyze_folder_recursive("donnees/faux")

# print("Résultats Vrais:", stats_vrais)
# print("Résultats Faux:", stats_faux)

In [3]:

print("real :", analyze_folder_recursive("../deepfakes_detection_datasets/ConfDF_teams/real"))
print("fake :", analyze_folder_recursive("../deepfakes_detection_datasets/ConfDF_teams/fake"))

real : {'fichiers_analyses': 42, 'avg_bitrate_kbps': np.float64(1320.7428095238095), 'std_bitrate_kbps': np.float64(86.47369246392032), 'codecs': {'h264': 42}}
fake : {'fichiers_analyses': 42, 'avg_bitrate_kbps': np.float64(1295.4079761904761), 'std_bitrate_kbps': np.float64(89.9788831661783), 'codecs': {'h264': 42}}
